# Wahlteilnehmer (`Contestant`)

`Contestant` repräsentiert einen Wahlteilnehmer — entweder eine Einzelpartei oder eine Koalition. Die Klasse ist immutable; Instanzen werden ausschließlich über Fabrikmethoden erzeugt.

In [ ]:
import pandas as pd
from ipres import Election, ElectionConfig, ConstituenciesConfig, SuperMajorityMargin, MarginUnit
from ipres.contestant import Contestant, contestantsFromParties, contestantsDictFromParties

---

## Einzelne Partei

In [ ]:
a = Contestant.from_party('A')
print(a)

# Liste von Einzelparteien
alle = contestantsFromParties(['A', 'B', 'C'])
print(alle)

# Dict von Einzelparteien
als_dict = contestantsDictFromParties(['A', 'B', 'C'])
print(als_dict)

In [ ]:
print(a.name)                              # 'A'
print(a.isSingleParty())                   # True
print(a.isCoalition())                     # False
print(a.members)                           # {} (leer)
print(a.getContainedParties())             # ['A']
print(a.getMemberVoteWeightsForParties())  # {'A': 1.0}

---

## Koalition

In [ ]:
a = Contestant.from_party('A')
b = Contestant.from_party('B')

# Automatische Gleichgewichtung (1/n pro Mitglied)
ab = Contestant.as_coalition('A+B', [a, b])
print(ab)
print(ab.member_vote_weights)  # {'A': 0.5, 'B': 0.5}

In [ ]:
# Explizite Gewichte: A hatte 60 %, B 40 % der Koalitionsstimmen
ab_weighted = Contestant.as_coalition('A+B', [a, b], member_vote_weights={'A': 0.6, 'B': 0.4})
print(ab_weighted.member_vote_weights)
print(ab_weighted.getMemberVoteWeight('A'))  # 0.6

In [ ]:
print(ab.getMemberNames())       # ['A', 'B']
print(ab.getMemberList())        # [Contestant(party='A'), Contestant(party='B')]
print(ab.getContainedParties())  # ['A', 'B']

---

## Verschachtelte Koalition

Koalitionen können aus anderen Koalitionen zusammengesetzt werden. `getMemberVoteWeightsForParties()` berechnet die Gewichte der Blattparteien dabei rekursiv, so dass die Summe immer 1,0 ergibt.

In [ ]:
c = Contestant.from_party('C')
ab = Contestant.as_coalition('A+B', [a, b])  # A=0.5, B=0.5

# (A+B) brachte 60 %, C 40 % der Gesamtstimmen
abc = Contestant.as_coalition('A+B+C', [ab, c], member_vote_weights={'A+B': 0.6, 'C': 0.4})

print(abc.getMemberVoteWeightsForParties())
# {'A': 0.3, 'B': 0.3, 'C': 0.4}

---

## Koalitionsbildung in der Wahl

Während eines Wahlgangs kann über `Ballot.formCoalition()` eine Koalition gebildet werden. Erreicht sie die Mehrheitsschwelle, gilt die Wahl als gewonnen — ohne dass ein weiterer Wahlgang nötig ist.

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1'],
    'constituency_size': [100_000],
}))
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    seed=42,
)
election = Election(electionConfig=config)
runde1 = election.start(votes={'WK1': {'A': 30, 'B': 28, 'C': 25, 'D': 17}})

print("Gewinner nach Wahlgang?", runde1.hasWinner())  # False

contestants = runde1.getContestants()
runde1.formCoalition("A+B", [contestants['A'], contestants['B']])

print("Gewinner nach Koalition?", election.hasWinner())  # True
print("Gewinner:", election.getWinner().name)            # A+B

---

## Zusammenfassung

| Methode / Attribut | Beschreibung |
|---|---|
| `Contestant.from_party(name)` | Einzelpartei erzeugen |
| `Contestant.as_coalition(name, members, weights?)` | Koalition erzeugen |
| `contestantsFromParties(names)` | Liste von Einzelparteien |
| `contestantsDictFromParties(names)` | Dict von Einzelparteien |
| `.name` | Anzeigename |
| `.isSingleParty()` / `.isCoalition()` | Typ prüfen |
| `.members` | Direkte Mitglieder (leer bei Einzelpartei) |
| `.member_vote_weights` | Stimmgewichte der direkten Mitglieder |
| `.getMemberList()` / `.getMemberNames()` | Mitgliederliste |
| `.getContainedParties()` | Alle Blattparteien (rekursiv) |
| `.getMemberVoteWeightsForParties()` | Blattpartei-Gewichte (rekursiv, Summe = 1,0) |
| `.getMemberVoteWeight(name)` | Gewicht eines direkten Mitglieds |
| `Ballot.formCoalition(name, members)` | Koalition während der Wahl bilden |